# M13 on Kaggle: E7, horizontal federated GBDT (dev check, then the full pool)

**CPU only: no GPU is needed, so this doesn't use GPU quota.** Accelerator: **None**. Internet *On*.

**Attach as input:** the same private Kaggle dataset as `m3_kaggle.ipynb` (the five files of the M1 `processed/` folder).

**Run:** *Save Version → Save & Run All (Commit)*.

Each window's fit is a real federated run: one XGBoost worker process per client (15), summing histograms through XGBoost's collective. The dev pool runs first as a pipeline check, then the full pool. Every step prints an exit code: 137 means the process was killed (usually out of memory).

**Output:** `/kaggle/working/m13_outputs.zip` with `reports/m13/` and the run folders. Bring it back, also after a failure.

In [ ]:
SEED = 42
import os, subprocess, sys
REPO = "/kaggle/working/IDS-code"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Indrajit147/IDS-code.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
# --no-deps: keep Kaggle's preinstalled stack intact; versions are recorded per run.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO, "--no-deps"], check=True)
os.chdir(REPO)
sys.path.insert(0, f"{REPO}/src")   # this kernel started before the editable install, so it cannot see it
os.environ["IDS_REPO_ROOT"] = REPO
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
assert os.path.exists("scripts/m13_fedgbdt.py"), "This clone predates M13 (no scripts/m13_fedgbdt.py). Restart the session so it clones the latest main."
import glob
assert glob.glob("/kaggle/input/**/data_manifest.json", recursive=True), "The processed dataset is not attached (no data_manifest.json under /kaggle/input): Add Input -> your M1 processed dataset."


In [ ]:
# Rebuild the M2 partitions (deterministic, a few minutes)
!python scripts/m2_build_partitions.py --env kaggle --pools dev full
# M2 redraws these figures, and their bytes differ on Kaggle. Nothing reads them. A changed report or JSON stays flagged.
!git checkout -- 'reports/m2/*/*.png'

In [ ]:
import glob, os, shutil
def pack():
    stage = "/kaggle/working/m13_stage"
    shutil.rmtree(stage, ignore_errors=True)
    os.makedirs(stage)
    if os.path.isdir("reports/m13"):
        shutil.copytree("reports/m13", f"{stage}/reports/m13")
    for r in sorted(glob.glob("/kaggle/working/results/runs/*_m13_fedgbdt_*")):
        shutil.copytree(r, f"{stage}/results/runs/{os.path.basename(r)}")
    shutil.make_archive("/kaggle/working/m13_outputs", "zip", stage)
    shutil.rmtree(stage)
    print("zip written:", sorted(glob.glob("reports/m13/*/*.md")))

In [ ]:
# dev first (pipeline check), then full; each in its own process, at 4 threads in all
for pool in ["dev", "full"]:
    print("=====", pool, flush=True)
    !python -X faulthandler scripts/m13_fedgbdt.py --env kaggle --pool {pool} --set resources.n_jobs=4 seed={SEED}; echo "exit code: $?"
    pack()
    if os.path.exists(f"reports/m13/{pool}/FEDGBDT.md"):
        print(open(f"reports/m13/{pool}/FEDGBDT.md", encoding="utf-8").read())